<a href="https://colab.research.google.com/github/harsh3-web/Ecommerce-Recommender/blob/main/_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
##Repo and libraries
!git clone https://github.com/harsh3-web/Ecommerce-Recommender.git
%cd Ecommerce-Recommender
!pip install -q torch_geometric flask

Cloning into 'Ecommerce-Recommender'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 31 (delta 13), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 70.39 KiB | 2.43 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/Ecommerce-Recommender
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/Ecommerce-Recommender-data"

!cp "{DATA}"/*.csv "{DATA}"/*.npy "{DATA}"/multimodal_gcn.pt .   # data, embeddings, graph, weights
!mkdir -p static/images templates                                  # Flask's standard folders
!cp "{DATA}"/images/*.jpg static/images/                           # Flask serves /static/ automatically
!ls *.npy *.pt; ls static/images | wc -l                           # 3 .npy, 1 .pt, 200 images

Mounted at /content/drive
edge_index.npy	image_emb.npy  multimodal_gcn.pt  text_emb.npy
200


In [3]:
##Imports
%%writefile recommender.py
"""recommender.py - loads everything ONCE at startup and serves recommendations for the Flask app."""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from torch_geometric.nn import GCNConv

Writing recommender.py


In [4]:
##Load data and embeddings
%%writefile -a recommender.py

products   = pd.read_csv("products_expanded.csv")            # rows in product_id order 1..200
browsing   = pd.read_csv("browsing_history_expanded.csv")
purchases  = pd.read_csv("purchases_expanded.csv")
n_users    = len(pd.read_csv("users_expanded.csv"))
n_products = len(products)
image_emb  = np.load("image_emb.npy")                        # (200, 2048) ResNet50, from notebook 05
text_emb   = np.load("text_emb.npy")                         # (200, 384) MiniLM, from notebook 05

Appending to recommender.py


In [5]:
##Interactions and item similarity, using ALL data
%%writefile -a recommender.py

BROWSE_W, BUY_W = 1.0, 4.0                                   # same weights as in evaluation
b = browsing[["user_id", "product_id"]].assign(weight=BROWSE_W)
p = purchases[["user_id", "product_id"]].assign(weight=BUY_W)
inter = pd.concat([b, p]).groupby(["user_id", "product_id"], as_index=False)["weight"].sum()
R = csr_matrix((inter["weight"].values, (inter["user_id"].values - 1, inter["product_id"].values - 1)),
               shape=(n_users, n_products))
item_sim = cosine_similarity(R.T); np.fill_diagonal(item_sim, 0)
seen = inter.groupby("user_id")["product_id"].apply(set).to_dict()   # products each user already touched

Appending to recommender.py


In [7]:
##Item-CF and content scores
%%writefile -a recommender.py

def item_cf_scores(uid):
    """Products similar (by co-interaction) to what this user touched."""
    return R[uid - 1].toarray().ravel() @ item_sim

def content_scores(uid):
    """Cosine similarity between each product's text embedding and the user's taste vector."""
    mine = inter[inter["user_id"] == uid]
    w = mine["weight"].values[:, None]
    profile = (w * text_emb[mine["product_id"].values - 1]).sum(axis=0) / w.sum()
    return text_emb @ (profile / np.linalg.norm(profile))

Appending to recommender.py


In [8]:
#The hybrid, the final model
%%writefile -a recommender.py

def minmax(x):
    spread = x.max() - x.min()
    return (x - x.min()) / spread if spread > 0 else x * 0

def hybrid_scores(uid, alpha=0.75):
    """Final model: 75% item-CF + 25% content, each min-max scaled over UNSEEN products only."""
    unseen = ~np.isin(np.arange(1, n_products + 1), list(seen.get(uid, ())))
    out = np.full(n_products, -np.inf)                           # seen products get -infinity
    out[unseen] = alpha * minmax(item_cf_scores(uid)[unseen]) + (1 - alpha) * minmax(content_scores(uid)[unseen])
    return out

Appending to recommender.py


In [9]:
##The trained multimodal model's class
%%writefile -a recommender.py

class MultiModalRec(nn.Module):          # SAME architecture as notebook 07 (needed to load the weights)
    def __init__(self, n_users, n_products, dim=64, use_gcn=True):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_products, dim)
        self.img_proj = nn.Linear(2048, dim)
        self.txt_proj = nn.Linear(384, dim)
        self.gcn = GCNConv(dim, dim) if use_gcn else None

    def product_vectors(self, img, txt, edge_index):
        x = self.item_emb.weight + self.img_proj(img) + self.txt_proj(txt)
        return x + self.gcn(x, edge_index) if self.gcn is not None else x

Appending to recommender.py


In [10]:
##Load the trained weights and precompute all scores
%%writefile -a recommender.py

mm_model = MultiModalRec(n_users, n_products)
mm_model.load_state_dict(torch.load("multimodal_gcn.pt"))      # the weights trained in notebook 07
mm_model.eval()                                                # inference mode
with torch.no_grad():
    prod = mm_model.product_vectors(torch.tensor(image_emb, dtype=torch.float32),
                                    torch.tensor(text_emb, dtype=torch.float32),
                                    torch.tensor(np.load("edge_index.npy"), dtype=torch.long))
    MM_SCORES = (mm_model.user_emb.weight @ prod.T).numpy()   # (500, 200): every user x product, once

Appending to recommender.py


In [11]:
##Popular scores and the main recommend() function
%%writefile -a recommender.py

pop_scores = purchases["product_id"].value_counts().reindex(products["product_id"], fill_value=0).values.astype(float)

METHODS = {"hybrid": hybrid_scores, "item_cf": item_cf_scores, "content": content_scores,
           "multimodal": lambda uid: MM_SCORES[uid - 1], "popular": lambda uid: pop_scores}

def recommend(uid, method="hybrid", k=10):
    """Top-k unseen products for a user. Unknown users fall back to 'popular' (cold start)."""
    if uid not in seen:
        method = "popular"
    recs = products.copy()
    recs["score"] = METHODS[method](uid)
    recs = recs[~recs["product_id"].isin(seen.get(uid, set()))]     # never re-recommend seen items
    return recs.sort_values("score", ascending=False).head(k), method

Appending to recommender.py


In [12]:
##Testing it before building the website
import importlib, recommender
importlib.reload(recommender)          # reload in case you re-ran a cell above

recs, used = recommender.recommend(7, "hybrid")
print("Method used:", used)
print(recs[["product_id", "product_name", "category", "score"]])

print("\nNew user 9999 ->", recommender.recommend(9999)[1])     # should print 'popular'

Method used: hybrid
     product_id                               product_name category     score
117         118  Indigo Nation Men Stripes Red Grey Shirts   Shirts  0.949779
23           24               Nyk Women Grey Vespa Handbag     Bags  0.895707
100         101               Wrangler Men Thin Blue Shirt   Shirts  0.748099
57           58          Wildcraft Men Black Fleece Jacket  Jackets  0.742549
95           96          Locomotive Men Washed Black Jeans    Jeans  0.737703
7             8       Murcia Women Lambretta Grey Handbags     Bags  0.722423
75           76                Jealous 21 Women Pink Jeans    Jeans  0.695967
1             2               Peperone Women Beige Handbag     Bags  0.694450
87           88              Flying Machine Men Blue Jeans    Jeans  0.686634
18           19     Baggit Women Motto Taj Brown Sling Bag     Bags  0.661684

New user 9999 -> popular


In [13]:
##The Flask app
%%writefile app.py
"""app.py - Flask web app: enter a user ID, choose a method, see top-10 recommendations."""
from flask import Flask, render_template, request
from recommender import recommend, METHODS, n_users

app = Flask(__name__)

@app.route("/")                                       # the home page URL
def index():
    return render_template("index.html", methods=list(METHODS), n_users=n_users)

Overwriting app.py


In [15]:
##the results page
%%writefile -a app.py

@app.route("/recommend")
def show_recommendations():
    uid = request.args.get("user_id", type=int)        # read ?user_id=7 from the URL
    method = request.args.get("method", "hybrid")
    if uid is None or uid < 1 or method not in METHODS:   # validate input before using it
        return render_template("index.html", methods=list(METHODS), n_users=n_users,
                               error="Please enter a valid user ID and method.")
    recs, used = recommend(uid, method)
    return render_template("recommendations.html", recs=recs.to_dict("records"), user_id=uid, method=used)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)                 # debug OFF (the original used debug=True: unsafe)

Appending to app.py


In [17]:
#the form
%%writefile templates/index.html
<!doctype html>
<html><head><title>Product Recommender</title>
<style>body{font-family:sans-serif;max-width:720px;margin:40px auto;padding:0 16px} label{display:block;margin:12px 0}</style>
</head><body>
<h1>Product Recommender</h1>
{% if error %}<p style="color:#b00">{{ error }}</p>{% endif %}
<form action="/recommend" method="get">
  <label>User ID (1-{{ n_users }}, or e.g. 9999 for a new user): <input type="number" name="user_id" min="1" required></label>
  <label>Method: <select name="method">{% for m in methods %}<option value="{{ m }}">{{ m }}</option>{% endfor %}</select></label>
  <button type="submit">Recommend</button>
</form>
</body></html>

Writing templates/index.html


In [18]:
##the results grid
%%writefile templates/recommendations.html
<!doctype html>
<html><head><title>Recommendations</title>
<style>body{font-family:sans-serif;max-width:900px;margin:40px auto;padding:0 16px}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(160px,1fr));gap:16px}
.card{border:1px solid #ddd;border-radius:8px;padding:8px;text-align:center}
.card img{width:100%;height:160px;object-fit:contain}</style>
</head><body>
<h1>Top 10 for user {{ user_id }}</h1>
<p>Method used: <b>{{ method }}</b> &middot; <a href="/">Back</a></p>
<div class="grid">
{% for r in recs %}
  <div class="card"><img src="/static/images/{{ r.product_id }}_1_front.jpg" alt="{{ r.product_name }}">
  <p>{{ r.product_name }}</p><small>{{ r.category }} &middot; Rs {{ r.price }} &middot; {{ r.rating }}/5</small></div>
{% endfor %}
</div></body></html>

Writing templates/recommendations.html


In [24]:
%%writefile static/style.css
/* Colour variables: change a colour here and it updates everywhere */
:root { --bg:#f6f7fb; --card:#ffffff; --text:#1f2330; --muted:#6b7080; --border:#e3e5ec; --accent:#4f46e5; --accent-text:#ffffff; }
@media (prefers-color-scheme: dark) {      /* automatically used when the system is in dark mode */
  :root { --bg:#14161c; --card:#1d2029; --text:#e8e9ee; --muted:#9ba0ad; --border:#2c303b; --accent:#818cf8; --accent-text:#14161c; }
}
* { box-sizing: border-box; }
body { margin:0; font-family:system-ui,-apple-system,"Segoe UI",Roboto,sans-serif; background:var(--bg); color:var(--text); line-height:1.5; }
.container { max-width:960px; margin:0 auto; padding:32px 16px; }
h1 { margin:0 0 4px; font-size:28px; }
.subtitle { margin:0 0 24px; color:var(--muted); }

/* Form page */
.panel { background:var(--card); border:1px solid var(--border); border-radius:12px; padding:24px; max-width:480px; }
label { display:block; font-weight:600; margin:16px 0 6px; }
input, select { width:100%; padding:10px 12px; border:1px solid var(--border); border-radius:8px; background:var(--bg); color:var(--text); font-size:15px; }
button, .btn { display:inline-block; margin-top:20px; padding:10px 18px; border:0; border-radius:8px; background:var(--accent); color:var(--accent-text); font-size:15px; font-weight:600; cursor:pointer; text-decoration:none; }
button:hover, .btn:hover { opacity:.9; }
.hint { font-size:13px; color:var(--muted); margin-top:6px; }
.error { background:#fde8e8; color:#9b1c1c; padding:10px 12px; border-radius:8px; }

/* Results page */
.toolbar { display:flex; align-items:center; justify-content:space-between; flex-wrap:wrap; gap:12px; margin-bottom:20px; }
.badge { display:inline-block; padding:2px 10px; border-radius:999px; background:var(--accent); color:var(--accent-text); font-size:13px; font-weight:600; }
.grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(170px,1fr)); gap:16px; }
.card { background:var(--card); border:1px solid var(--border); border-radius:12px; overflow:hidden; transition:transform .15s; }
.card:hover { transform:translateY(-3px); }
.card img { width:100%; height:180px; object-fit:contain; background:#ffffff; padding:8px; }
.info { padding:10px 12px; }
.rank { font-size:12px; color:var(--muted); }
.name { margin:2px 0 6px; font-size:14px; font-weight:600; }
.meta { display:flex; justify-content:space-between; font-size:13px; color:var(--muted); }

Writing static/style.css


In [25]:
%%writefile templates/index.html
<!doctype html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Product Recommender</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head><body><div class="container">
  <h1>Product Recommender</h1>
  <p class="subtitle">Personalised fashion recommendations from 200 real Myntra products</p>
  <div class="panel">
    {% if error %}<p class="error">{{ error }}</p>{% endif %}
    <form action="/recommend" method="get">
      <label for="uid">User ID</label>
      <input id="uid" type="number" name="user_id" min="1" placeholder="e.g. 7" required>
      <p class="hint">Existing users: 1-{{ n_users }}. Try 9999 to see a new user (cold start).</p>
      <label for="method">Method</label>
      <select id="method" name="method">
        {% for m in methods %}<option value="{{ m }}">{{ m }}</option>{% endfor %}
      </select>
      <button type="submit">Get recommendations</button>
    </form>
  </div>
</div></body></html>

Overwriting templates/index.html


In [26]:
%%writefile templates/recommendations.html
<!doctype html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Recommendations for user {{ user_id }}</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head><body><div class="container">
  <div class="toolbar">
    <div><h1>Top 10 for user {{ user_id }}</h1>
      <span class="badge">{{ method }}</span></div>
    <a class="btn" href="/">&larr; New search</a>
  </div>
  <div class="grid">
    {% for r in recs %}
    <div class="card">
      <img src="{{ url_for('static', filename='images/' ~ r.product_id ~ '_1_front.jpg') }}" alt="{{ r.product_name }}">
      <div class="info">
        <div class="rank">#{{ loop.index }} &middot; {{ r.category }}</div>
        <p class="name">{{ r.product_name }}</p>
        <div class="meta"><span>Rs {{ r.price }}</span><span>&#9733; {{ r.rating }}</span></div>
      </div>
    </div>
    {% endfor %}
  </div>
</div></body></html>

Overwriting templates/recommendations.html


In [35]:
import os, subprocess, time, urllib.request
from google.colab import output

os.chdir("/content/Ecommerce-Recommender")                         # always run from the project folder
subprocess.run("pkill -f app.py; fuser -k 5000/tcp", shell=True)     # stop ANY old server on port 5000
time.sleep(2)

log = open("app_log.txt", "w")
server = subprocess.Popen(["python", "app.py"], stdout=log, stderr=log)   # start Flask in the background

for _ in range(60):                                                # wait up to 60 seconds for it to answer
    try:
        page = urllib.request.urlopen("http://127.0.0.1:5000").read().decode()
        break                                                      # it answered -> ready
    except Exception:
        if server.poll() is not None:                              # the process died -> show why
            print(open("app_log.txt").read())
            raise SystemExit("Server crashed - see the log above")
        time.sleep(1)
else:
    raise SystemExit("Server did not start within 60 seconds - check app_log.txt")

print("Server is running | new design loaded:", "style.css" in page)   # should print True
output.serve_kernel_port_as_window(5000)                          # prints the clickable link

Server is running | new design loaded: True
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>